<a href="https://colab.research.google.com/github/Vigneshvar16/CSA0607-DAA/blob/main/CO2_AT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import math

def sort_alerts(alerts):
    for i in range(1, len(alerts)):
        key = alerts[i]
        j = i - 1
        while j >= 0 and alerts[j][1] < key[1]:   # fix 1: j >= 0
            alerts[j + 1] = alerts[j]
            j -= 1
        alerts[j + 1] = key                        # fix 2: j + 1
    return alerts

def is_blocked(blocklist, ip):
    for i in range(len(blocklist)):
        if blocklist[i] == ip:
            return True                            # fix 3: True on match
    return False                                   # fix 4: only after full scan

def find_signature(log, pattern):
    n, m = len(log), len(pattern)
    hits = []
    if m == 0:
        return hits
    for i in range(n - m + 1):                     # fix 5: last window included
        for j in range(m):
            if log[i + j] != pattern[j]:
                break
        else:                                      # fix 6: only if no mismatch
            hits.append(i)
    return hits

def closest_logins(events):
    min_dist = math.inf                            # fix 7: not 0
    pair = None
    for i in range(len(events)):
        for j in range(i + 1, len(events)):        # fix 9: no self/repeat pairs
            d = math.sqrt((events[i][0] - events[j][0]) ** 2 +
                          (events[i][1] - events[j][1]) ** 2)
            if d < min_dist:                       # fix 8: < for minimum
                min_dist = d
                pair = (i, j)
    return pair, min_dist

# ---- re-test on the given data + boundary cases ----
alerts = [("A1", 3), ("A2", 9), ("A3", 5), ("A4", 9), ("A5", 1)]
assert sort_alerts(alerts[:]) == [("A2", 9), ("A4", 9), ("A3", 5), ("A1", 3), ("A5", 1)]
assert sort_alerts([]) == [] and sort_alerts([("X", 2)]) == [("X", 2)]
assert sort_alerts([("a", 4), ("b", 4), ("c", 4)]) == [("a", 4), ("b", 4), ("c", 4)]
bl = ["10.0.0.5", "10.0.0.9", "172.16.4.2"]
assert [is_blocked(bl, q) for q in ["10.0.0.9", "10.0.0.5", "8.8.8.8"]] == [True, True, False]
assert is_blocked([], "1.1.1.1") is False
assert find_signature("FAILFAIXFAIL", "FAIL") == [0, 8]
assert find_signature("XXFAIL", "FAIL") == [2]            # pattern at very end
assert find_signature("FA", "FAIL") == [] and find_signature("", "A") == []
ev = [(2, 3), (12, 30), (40, 50), (5, 1), (12, 10), (3, 4)]
p, d = closest_logins(ev)
assert p == (0, 5) and abs(d - math.sqrt(2)) < 1e-9
assert closest_logins([]) == (None, math.inf) and closest_logins([(1, 1)]) == (None, math.inf)
print("all corrected tests passed")


all corrected tests passed


In [4]:
def sort_alerts(alerts):
    alerts.sort(key=lambda a: -a[1])      # Timsort: stable, O(n log n), O(n) if nearly sorted
    return alerts

def is_blocked(blockset, ip):             # blockset = set(blocklist), built once
    return ip in blockset                 # O(1) average per lookup

def find_signature(log, pattern):
    n, m = len(log), len(pattern)
    if m == 0:
        return []
    hits = []
    for i in range(n - m + 1):
        for j in range(m):
            if log[i + j] != pattern[j]:  # stop at first mismatch: no wasted comparisons
                break
        else:
            hits.append(i)
    return hits

def closest_logins(events):
    n, best, pair = len(events), math.inf, None
    order = sorted(range(n), key=lambda k: events[k][0])    # sort by x once
    for a in range(n):
        i = order[a]
        xi, yi = events[i]
        for b in range(a + 1, n):                           # each pair once, no self-pairs
            j = order[b]
            dx = events[j][0] - xi
            if dx * dx > best:                              # prune: x-gap alone exceeds best
                break
            dy = events[j][1] - yi
            d2 = dx * dx + dy * dy                          # squared distance, no sqrt
            p = (min(i, j), max(i, j))
            if d2 < best or (d2 == best and p < pair):      # same tie-break as brute force
                best, pair = d2, p
    return pair, math.sqrt(best)


In [5]:
import random, math

def brute_count(ev):                       # corrected brute force, counts distance computations
    c = 0
    for i in range(len(ev)):
        for j in range(i + 1, len(ev)):
            math.sqrt((ev[i][0]-ev[j][0])**2 + (ev[i][1]-ev[j][1])**2); c += 1
    return c

def opt_count(ev):                         # optimised version, counts distance computations
    best, c = math.inf, 0
    s = sorted(ev)
    for a in range(len(s)):
        for b in range(a + 1, len(s)):
            dx = s[b][0] - s[a][0]
            if dx * dx > best: break
            c += 1
            best = min(best, dx*dx + (s[b][1]-s[a][1])**2)
    return c

def insertion_count(alerts):               # counts severity comparisons
    a, c = alerts[:], 0
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0:
            c += 1
            if not a[j][1] < key[1]: break
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = key
    return c

random.seed(7)
print("n      brute        optimised")
for n in (1000, 4000, 8000):
    ev = [(random.uniform(0, 1000), random.uniform(0, 1000)) for _ in range(n)]
    print(f"{n:<6} {brute_count(ev):<12} {opt_count(ev)}")

n = 2000
rand = [(i, random.randint(1, 10)) for i in range(n)]
near = sorted(rand, key=lambda a: -a[1])
for _ in range(20):                         # 20 random swaps -> nearly sorted
    x, y = random.randrange(n), random.randrange(n); near[x], near[y] = near[y], near[x]
print("insertion comparisons  random:", insertion_count(rand), " nearly sorted:", insertion_count(near))


n      brute        optimised
1000   499500       950
4000   7998000      8471
8000   31996000     12910
insertion comparisons  random: 908297  nearly sorted: 29623


In [6]:
"""SentinelOps SOC alert-triage module (corrected and optimised)."""
import math
import random


def sort_alerts(alerts):
    """Sort (id, severity) alerts by severity, highest first.

    Stable: equal severities keep arrival order. Sorts in place and returns the list.
    Time: O(n log n) worst case, O(n) on nearly sorted input (Timsort).
    """
    alerts.sort(key=lambda a: -a[1])
    return alerts


def is_blocked(blockset, ip):
    """Return True if ip is in blockset (a set built once via set(blocklist)). O(1) average."""
    return ip in blockset


def find_signature(log, pattern):
    """Return every start index of pattern in log (brute force, O(n*m) worst case).

    Windows run over i = 0..n-m so a match at the very end is found; the inner
    loop stops at the first mismatch. Empty or too-long patterns give [].
    """
    n, m = len(log), len(pattern)
    if m == 0:
        return []
    hits = []
    for i in range(n - m + 1):
        for j in range(m):
            if log[i + j] != pattern[j]:
                break
        else:                       # reached only when no mismatch occurred
            hits.append(i)
    return hits


def closest_logins(events):
    """Return ((i, j), distance) for the closest pair of (x, y) points, i < j.

    Points are sorted by x; the inner loop stops once the x-gap squared exceeds the
    best squared distance. sqrt is taken once at the end. Fewer than two points
    give (None, inf). Ties resolve to the lexicographically smallest index pair.
    """
    n, best, pair = len(events), math.inf, None
    order = sorted(range(n), key=lambda k: events[k][0])
    for a in range(n):
        i = order[a]
        xi, yi = events[i]
        for b in range(a + 1, n):
            j = order[b]
            dx = events[j][0] - xi
            if dx * dx > best:
                break
            dy = events[j][1] - yi
            d2 = dx * dx + dy * dy
            p = (min(i, j), max(i, j))
            if d2 < best or (d2 == best and p < pair):
                best, pair = d2, p
    return pair, math.sqrt(best)


def _brute(events):
    """Reference brute-force closest pair, used only by the tests."""
    best, pair = math.inf, None
    for i in range(len(events)):
        for j in range(i + 1, len(events)):
            d = math.hypot(events[i][0] - events[j][0], events[i][1] - events[j][1])
            if d < best:
                best, pair = d, (i, j)
    return pair, best


def run_tests():
    """Check every function on the given data and the boundary cases."""
    alerts = [("A1", 3), ("A2", 9), ("A3", 5), ("A4", 9), ("A5", 1)]
    assert sort_alerts(alerts[:]) == [("A2", 9), ("A4", 9), ("A3", 5), ("A1", 3), ("A5", 1)]
    assert sort_alerts([]) == [] and sort_alerts([("X", 2)]) == [("X", 2)]
    assert sort_alerts([("a", 4), ("b", 4), ("c", 4)]) == [("a", 4), ("b", 4), ("c", 4)]

    blockset = set(["10.0.0.5", "10.0.0.9", "172.16.4.2"])
    assert [is_blocked(blockset, q) for q in ("10.0.0.9", "10.0.0.5", "8.8.8.8")] == [True, True, False]
    assert is_blocked(set(), "1.1.1.1") is False            # empty blocklist

    assert find_signature("FAILFAIXFAIL", "FAIL") == [0, 8]
    assert find_signature("XXFAIL", "FAIL") == [2]          # pattern at the very end
    assert find_signature("FA", "FAIL") == []               # pattern longer than log
    assert find_signature("AAAA", "AA") == [0, 1, 2]        # overlapping matches
    assert find_signature("", "A") == [] and find_signature("ABC", "") == []

    ev = [(2, 3), (12, 30), (40, 50), (5, 1), (12, 10), (3, 4)]
    pair, d = closest_logins(ev)
    assert pair == (0, 5) and abs(d - math.sqrt(2)) < 1e-9
    assert closest_logins([]) == (None, math.inf)           # fewer than two events
    assert closest_logins([(1, 1)]) == (None, math.inf)
    assert closest_logins([(0, 0), (0, 0)]) == ((0, 1), 0.0)  # duplicate points
    random.seed(1)
    for _ in range(200):                                    # cross-check against brute force
        pts = [(random.randint(0, 30), random.randint(0, 30)) for _ in range(random.randint(2, 40))]
        assert closest_logins(pts) == _brute(pts)
    print("All tests passed")


if __name__ == "__main__":
    run_tests()


All tests passed
